# SciQ Builder

Build `science_nature_facts_sciq_v1` from `allenai/sciq`.

This notebook keeps the first facts dataset deliberately small and auditable:

- `text` is only the SciQ `support` passage
- questions and answers are kept as metadata for analysis/debug, not as the main retrieval text
- taxonomy is lightweight: `subject`, `topic`, `source_type`, `taxonomy_source`, `taxonomy_confidence`
- output is saved to `Datasets/science_nature_facts_sciq_v1`


In [ ]:
# @title Mount Google Drive

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/NLP")

if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/NLP")

if not PROJECT_ROOT.exists():
    raise FileNotFoundError(
        "Non trovo la cartella progetto. "
        "Controlla se il path è /content/drive/MyDrive/NLP oppure modifica PROJECT_ROOT manualmente."
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Exists:", PROJECT_ROOT.exists())

## 0. Setup

If you run this in Colab and dependencies are missing, uncomment the install cell below.

In [ ]:
# !pip -q install datasets pandas

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import re
import shutil

import pandas as pd
from datasets import Dataset, DatasetDict, load_dataset, load_from_disk

# Example for Colab/Drive:
# PROJECT_ROOT_OVERRIDE = "/content/drive/MyDrive/NLP"
PROJECT_ROOT_OVERRIDE = None

PROJECT_MARKERS = [Path("requirements.txt"), Path("millionaire_client")]
PROJECT_ROOT_CANDIDATES = [
    Path.cwd(),
    Path("/content/drive/MyDrive/NLP"),
    Path("/content/drive/MyDrive/Colab Notebooks/NLP"),
    Path("/gdrive/MyDrive/NLP"),
    Path("/gdrive/MyDrive/Colab Notebooks/NLP"),
]

def looks_like_project_root(path):
    return any((path / marker).exists() for marker in PROJECT_MARKERS)

if PROJECT_ROOT_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser()
else:
    PROJECT_ROOT = next((candidate for candidate in PROJECT_ROOT_CANDIDATES if looks_like_project_root(candidate)), Path.cwd())

DATASETS_DIR = PROJECT_ROOT / "Datasets"
LOGS_DIR = PROJECT_ROOT / "logs"
DATASETS_DIR.mkdir(parents=True, exist_ok=True)
LOGS_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_DATASET = "allenai/sciq"
SOURCE_LICENSE = "cc-by-nc-3.0"
SOURCE_TYPE = "sciq_support"
TAXONOMY_SOURCE = "sciq_rule_based_v1"

OUTPUT_DATASET_DIR = DATASETS_DIR / "science_nature_facts_sciq_v1"
BUILD_REPORT_JSON = LOGS_DIR / "science_nature_facts_sciq_v1_build_report.json"
AUDIT_CSV = LOGS_DIR / "science_nature_facts_sciq_v1_audit.csv"
DUPLICATES_CSV = LOGS_DIR / "science_nature_facts_sciq_v1_duplicate_supports.csv"

# Leave False to avoid accidental overwrite. Set True only when intentionally regenerating v1.
OVERWRITE_OUTPUT = False

print("project root:", PROJECT_ROOT)
print("output dataset:", OUTPUT_DATASET_DIR)
print("build report:", BUILD_REPORT_JSON)

## 1. Load SciQ

SciQ provides `question`, `correct_answer`, `distractor1`, `distractor2`, `distractor3`, and `support`. We use `support` as the retrieval text.

In [ ]:
raw = load_dataset(SOURCE_DATASET)

print(raw)
for split_name, split_ds in raw.items():
    print(split_name, len(split_ds), split_ds.column_names)

display(pd.DataFrame({"split": list(raw.keys()), "rows": [len(ds) for ds in raw.values()]}))

## 2. Lightweight Taxonomy

`subject` is controlled and small. `topic` is free text derived from `correct_answer`.

In [ ]:
SUBJECT_LABELS = [
    "physics",
    "chemistry",
    "biology",
    "ecology",
    "earth_science",
    "astronomy",
    "scientific_method",
    "general_science",
]

SUBJECT_KEYWORDS = {
    "physics": [
        "force", "motion", "speed", "velocity", "acceleration", "friction", "gravity", "energy",
        "mass", "weight", "pressure", "light", "sound", "wave", "electricity", "electric",
        "magnet", "magnetic", "current", "circuit", "heat", "temperature", "thermal", "momentum",
        "inertia", "machine",
    ],
    "chemistry": [
        "atom", "atomic", "electron", "proton", "neutron", "molecule", "molecular", "compound",
        "element", "chemical", "chemistry", "bond", "ionic", "covalent", "ion", "acid", "base",
        "solution", "solvent", "sodium", "chlorine", "chloride", "reaction", "reactant", "product",
        "metal", "nonmetal", "mixture", "periodic table",
    ],
    "biology": [
        "cell", "organism", "organ", "tissue", "dna", "gene", "genetic", "protein", "enzyme",
        "photosynthesis", "respiration", "plant", "animal", "bacteria", "virus", "mitochondria",
        "nucleus", "membrane", "reproduction", "evolution", "trait", "body", "blood", "muscle",
        "digestion", "nervous", "skeleton", "root", "leaf", "seed", "flower",
    ],
    "ecology": [
        "ecosystem", "ecology", "habitat", "population", "community", "predator", "prey",
        "food chain", "food web", "species", "environment", "resource", "producer", "consumer",
        "decomposer", "competition", "adaptation", "niche", "biodiversity", "biome", "desert",
        "forest", "aquatic", "pollution", "conservation",
    ],
    "earth_science": [
        "earth", "rock", "soil", "erosion", "weathering", "volcano", "earthquake", "mineral",
        "fossil", "sediment", "sedimentary", "igneous", "metamorphic", "plate tectonics",
        "atmosphere", "climate", "weather", "water cycle", "ocean", "river", "glacier", "mountain",
        "landform",
    ],
    "astronomy": [
        "planet", "star", "moon", "sun", "solar system", "orbit", "galaxy", "universe", "asteroid",
        "comet", "space", "telescope", "constellation", "astronomy",
    ],
    "scientific_method": [
        "experiment", "hypothesis", "variable", "observation", "measurement", "data", "evidence",
        "conclusion", "control group", "independent variable", "dependent variable", "model", "theory",
        "scientific method", "test", "investigate",
    ],
}

WHITESPACE_RE = re.compile(r"\s+")

def normalize_text(value):
    if value is None:
        return ""
    text = str(value).replace("\u00a0", " ")
    return WHITESPACE_RE.sub(" ", text).strip()

def normalize_topic(value):
    topic = normalize_text(value).lower()
    topic = re.sub(r"[^a-z0-9+\-/ ]+", "", topic)
    return WHITESPACE_RE.sub(" ", topic).strip() or "unknown"

def hash_text(text):
    return hashlib.sha1(normalize_text(text).lower().encode("utf-8")).hexdigest()

def keyword_hits(blob, keywords):
    hits = []
    for keyword in keywords:
        kw = normalize_text(keyword).lower()
        if not kw:
            continue
        if " " in kw:
            pattern = r"\b" + re.escape(kw) + r"\b"
        else:
            pattern = r"\b" + re.escape(kw) + r"s?\b"
        if re.search(pattern, blob):
            hits.append(keyword)
    return hits

def classify_subject(question, support, answer):
    blob = " ".join([normalize_text(question), normalize_text(support), normalize_text(answer)]).lower()
    scores = {}
    hit_map = {}
    for subject, keywords in SUBJECT_KEYWORDS.items():
        hits = keyword_hits(blob, keywords)
        scores[subject] = len(hits)
        hit_map[subject] = hits

    best_subject = max(scores, key=scores.get)
    best_score = scores[best_subject]

    if best_score == 0:
        return "general_science", 0.40, []

    tied_subjects = [subject for subject, score in scores.items() if score == best_score]
    confidence = 0.95 if best_score >= 2 and len(tied_subjects) == 1 else 0.75
    return best_subject, confidence, hit_map[best_subject]

print("subject labels:", SUBJECT_LABELS)
print("taxonomy source:", TAXONOMY_SOURCE)

## 3. Normalize Rows

Each SciQ row becomes one candidate document. Empty supports are dropped. Duplicate supports are logged and deduplicated.

In [ ]:
rows = []
empty_support_rows = []

for split_name, split_ds in raw.items():
    for row_idx, example in enumerate(split_ds):
        support = normalize_text(example.get("support", ""))
        question = normalize_text(example.get("question", ""))
        correct_answer = normalize_text(example.get("correct_answer", ""))
        distractor1 = normalize_text(example.get("distractor1", ""))
        distractor2 = normalize_text(example.get("distractor2", ""))
        distractor3 = normalize_text(example.get("distractor3", ""))

        if not support:
            empty_support_rows.append({"split": split_name, "row_idx": row_idx, "question": question})
            continue

        subject, confidence, taxonomy_hits = classify_subject(question, support, correct_answer)
        topic = normalize_topic(correct_answer)
        doc_id = f"sciq_{split_name}_{row_idx:06d}"

        rows.append({
            "doc_id": doc_id,
            "text": support,
            "source_dataset": SOURCE_DATASET,
            "source_split": split_name,
            "source_type": SOURCE_TYPE,
            "license": SOURCE_LICENSE,
            "question": question,
            "correct_answer": correct_answer,
            "distractor1": distractor1,
            "distractor2": distractor2,
            "distractor3": distractor3,
            "subject": subject,
            "topic": topic,
            "taxonomy_source": TAXONOMY_SOURCE,
            "taxonomy_confidence": confidence,
            "taxonomy_hits": " | ".join(taxonomy_hits),
            "support_hash": hash_text(support),
        })

df = pd.DataFrame(rows)
print("candidate rows:", len(df))
print("empty support rows:", len(empty_support_rows))
display(df.head())

In [ ]:
duplicate_mask = df.duplicated("support_hash", keep=False)
duplicates = df.loc[duplicate_mask].sort_values(["support_hash", "source_split", "doc_id"]).copy()
dedup_df = df.drop_duplicates("support_hash", keep="first").reset_index(drop=True)

print("rows before dedup:", len(df))
print("duplicate rows:", len(duplicates))
print("rows after dedup:", len(dedup_df))

if len(duplicates):
    duplicates.to_csv(DUPLICATES_CSV, index=False)
    print("saved duplicates:", DUPLICATES_CSV)
    display(duplicates[["doc_id", "source_split", "correct_answer", "subject", "text"]].head(20))

## 4. Audit Distribution

This is the main sanity check before saving. If `general_science` is too large, expand the keyword rules and rerun.

In [ ]:
subject_counts = dedup_df["subject"].value_counts().rename_axis("subject").reset_index(name="rows")
topic_counts = dedup_df["topic"].value_counts().head(30).rename_axis("topic").reset_index(name="rows")
confidence_counts = dedup_df["taxonomy_confidence"].value_counts().sort_index().rename_axis("confidence").reset_index(name="rows")

print("subject distribution")
display(subject_counts)
print("taxonomy confidence distribution")
display(confidence_counts)
print("top topics")
display(topic_counts)

audit_cols = [
    "doc_id", "source_split", "subject", "topic", "taxonomy_confidence",
    "taxonomy_hits", "correct_answer", "text",
]
display(dedup_df[audit_cols].sample(min(20, len(dedup_df)), random_state=42))

In [ ]:
general_sample = dedup_df[dedup_df["subject"] == "general_science"]
print("general_science rows:", len(general_sample))
if len(general_sample):
    display(general_sample[audit_cols].sample(min(20, len(general_sample)), random_state=7))

## 5. Save Dataset

The output is a single `train` split because this dataset is used as a retrieval corpus, not as a model training benchmark. Original SciQ split is preserved in `source_split`.

In [ ]:
if OUTPUT_DATASET_DIR.exists():
    if not OVERWRITE_OUTPUT:
        raise FileExistsError(
            f"Output already exists: {OUTPUT_DATASET_DIR}. Set OVERWRITE_OUTPUT = True to regenerate."
        )
    shutil.rmtree(OUTPUT_DATASET_DIR)

dataset = Dataset.from_pandas(dedup_df, preserve_index=False)
dataset_dict = DatasetDict({"train": dataset})
dataset_dict.save_to_disk(str(OUTPUT_DATASET_DIR))

dedup_df.to_csv(AUDIT_CSV, index=False)

report = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_dataset": SOURCE_DATASET,
    "source_license": SOURCE_LICENSE,
    "output_dataset_dir": str(OUTPUT_DATASET_DIR),
    "audit_csv": str(AUDIT_CSV),
    "duplicates_csv": str(DUPLICATES_CSV) if len(duplicates) else "",
    "source_splits": {split: len(ds) for split, ds in raw.items()},
    "candidate_rows": int(len(df)),
    "empty_support_rows": int(len(empty_support_rows)),
    "duplicate_rows": int(len(duplicates)),
    "final_rows": int(len(dedup_df)),
    "fields": list(dedup_df.columns),
    "subject_labels": SUBJECT_LABELS,
    "subject_counts": subject_counts.set_index("subject")["rows"].to_dict(),
    "top_topics": topic_counts.set_index("topic")["rows"].to_dict(),
    "taxonomy_source": TAXONOMY_SOURCE,
}

BUILD_REPORT_JSON.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

print("saved dataset:", OUTPUT_DATASET_DIR)
print("saved audit csv:", AUDIT_CSV)
print("saved build report:", BUILD_REPORT_JSON)
print(json.dumps(report, indent=2, ensure_ascii=False)[:3000])

## 6. Reload Check

Quick check that the saved artifact can be loaded by the future index builder.

In [ ]:
loaded = load_from_disk(str(OUTPUT_DATASET_DIR))
print(loaded)
display(loaded["train"].to_pandas().head(10))

## Next Step

Build `Indexes/science_nature_facts_sciq_v1` from this dataset. The index should use `text` as page content and keep at least these metadata fields: `doc_id`, `source_dataset`, `source_split`, `source_type`, `license`, `subject`, `topic`, `taxonomy_source`, `taxonomy_confidence`.